# RAG Indexing Pipeline — Run Once

Parses, chunks, embeds, and upserts `langchain-core` source into Pinecone.

> **Run this file once.** After upsert is complete, use `rag_query.ipynb` for retrieval and generation.

## Setup — Load credentials

In [ ]:
import os
from pathlib import Path

# Always run from project root regardless of where the notebook is launched from
PROJECT_ROOT = Path(__file__).parent.parent if "__file__" in dir() else Path.cwd().parent
os.chdir(PROJECT_ROOT)
print(f"Working directory: {os.getcwd()}")

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

PINECONE_API_KEY = os.environ["PINECONE_API_KEY"]
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
ANTHROPIC_API_KEY = os.environ["ANTHROPIC_API_KEY"]

print("Credentials loaded.")

---
## Module 1 — Load Raw Files
Walk `langchain/libs/core/langchain_core/` and load all `.py` files.

**Test:** Total file count + sample metadata.

In [ ]:
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser
from langchain_text_splitters import Language

loader = GenericLoader.from_filesystem(
    "langchain/libs/core/langchain_core/",
    glob="**/*.py",
    suffixes=[".py"],
    parser=LanguageParser(language=Language.PYTHON, parser_threshold=0),
)
raw_docs = loader.load()

print(f"Loaded {len(raw_docs)} documents")
print(raw_docs[0].metadata)

---
## Module 2 — Inspect AST Parsed Documents
Verify one document per function/class. Check `content_type` field.

**Test:** `content_type` should be `functions_classes` for most docs.

In [ ]:
for doc in raw_docs[:3]:
    print("Metadata:", doc.metadata)
    print("Content preview:")
    print(doc.page_content[:300])
    print("---")

# Count by content_type
from collections import Counter
type_counts = Counter(d.metadata.get("content_type", "unknown") for d in raw_docs)
print("\nDocument types:", dict(type_counts))

---
## Module 3 — Chunk + Attach Metadata
Split docs with `RecursiveCharacterTextSplitter` (Python mode, size=1000, overlap=200).
Attach `module`, `filename`, `package` metadata to every chunk.

**Test:** Chunk count, min/max/avg sizes. Max should stay well under 6000 chars.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language
import os

splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=1000,
    chunk_overlap=200,
)
chunks = splitter.split_documents(raw_docs)

for chunk in chunks:
    src = chunk.metadata.get("source", "")
    parts = src.replace("\\", "/").split("/")
    chunk.metadata["filename"] = parts[-1] if parts else ""
    chunk.metadata["module"] = parts[-2] if len(parts) >= 2 else ""
    chunk.metadata["package"] = "langchain-core"

sizes = [len(c.page_content) for c in chunks]
print(f"Total chunks : {len(chunks)}")
print(f"Min size     : {min(sizes)} chars")
print(f"Max size     : {max(sizes)} chars")
print(f"Avg size     : {sum(sizes) // len(sizes)} chars")
print(f"\nSample chunk metadata: {chunks[0].metadata}")

---
## Module 4 — Fit & Save BM25
Fit `BM25Encoder` on all chunk texts. Save to `bm25_params.json`.

**Test:** Reload encoder, encode a test query, verify non-zero sparse terms.

In [ ]:
from pinecone_text.sparse import BM25Encoder

bm25 = BM25Encoder()
bm25.fit([c.page_content for c in chunks])
bm25.dump("data/bm25_params.json")
print("BM25 fitted and saved to data/bm25_params.json")

In [ ]:
from pinecone_text.sparse import BM25Encoder

# Verify round-trip
bm25_loaded = BM25Encoder()
bm25_loaded.load("data/bm25_params.json")
test_vec = bm25_loaded.encode_queries("runnable chain invoke")
print(f"Sparse vector has {len(test_vec['indices'])} non-zero terms")

---
## Module 5 — Create Pinecone Index + Upsert
Create serverless Pinecone index (`dotproduct`, 1536 dims).  
Embed with OpenAI `text-embedding-3-small`. Upsert dense + sparse + metadata.

**Test:** `describe_index_stats()` vector count == chunk count.

> **Run once — costs money.**

In [ ]:
from langchain_openai import OpenAIEmbeddings
from pinecone import Pinecone, ServerlessSpec
import time

INDEX_NAME = "langchain-core-rag"

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=OPENAI_API_KEY,
)

pc = Pinecone(api_key=PINECONE_API_KEY)

if INDEX_NAME not in [i.name for i in pc.list_indexes()]:
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,
        metric="dotproduct",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print(f"Index '{INDEX_NAME}' created.")
else:
    print(f"Index '{INDEX_NAME}' already exists.")

index = pc.Index(INDEX_NAME)

BATCH = 100
for i in range(0, len(chunks), BATCH):
    batch = chunks[i : i + BATCH]
    texts = [c.page_content for c in batch]
    dense_vecs = embeddings.embed_documents(texts)
    sparse_vecs = [bm25_loaded.encode_documents(t) for t in texts]

    vectors = [
        {
            "id": f"chunk-{i + j}",
            "values": dense_vecs[j],
            "sparse_values": sparse_vecs[j],
            "metadata": {**batch[j].metadata, "text": texts[j]},
        }
        for j in range(len(batch))
    ]
    index.upsert(vectors=vectors)

    if (i // BATCH) % 10 == 0:
        print(f"Upserted {min(i + BATCH, len(chunks))}/{len(chunks)} chunks...")

time.sleep(2)  # let Pinecone settle
print("\nIndex stats:")
print(index.describe_index_stats())